## Result filter module - _Attention-Retrieval (AR)_ - Fine-tuning

Why this failed:
- The model has already been pre-trained on MARCO dataset, and further fine-tuning has led to overfitting. Generalization to unseen data is important for iRAT web data changes rapidly.

### Initialization

In [1]:
import sys
import os
if 'google.colab' in sys.modules:
	from google.colab import drive
	from IPython.display import clear_output
	!pip install --upgrade datasets sentence_transformers
	!pip install focal_loss_torch torch_optimizer
	clear_output()
	drive.mount('/content/drive')
	clear_output()
	# copy *.csv files from drive to current directory
	!cp drive/MyDrive/iRAT/*.csv .
	!cp drive/MyDrive/iRAT/*.py .
	!cp -r drive/MyDrive/iRAT/*_dataset .
	clear_output()
else:
	# if not in 'notebooks' directory, change to it
	if not os.getcwd().endswith('Result-filter-RL'):
		os.chdir('notebooks')
		os.chdir('Result-filter-RL')

os.environ['WANDB_DISABLED'] = 'true'  # disable Weights & Biases logging

# import numpy as np
# import random
# import torch

# # set random seeds for reproducibility
# seed = 42
# random.seed(seed)
# np.random.seed(seed)
# torch.manual_seed(seed)
# if torch.cuda.is_available():
# 	torch.cuda.manual_seed_all(seed)
# 	# for fully deterministic (seeded) CuDNN behavior (slower), you can also do:
# 	# torch.backends.cudnn.deterministic = True
# 	# torch.backends.cudnn.benchmark = False

### Load the model and dataset

In [ ]:
from AR_evaluate import evaluate_model  # from local file
from sentence_transformers import CrossEncoder

model_name = 'cross-encoder/ms-marco-MiniLM-L6-v2'  # 22.7M params
model_name_short = 'MiniLM-L6-v2'

# model_name = 'cross-encoder/ms-marco-MiniLM-L4-v2'  # 19.2M params
# model_name_short = 'MiniLM-L4-v2'


if model_name_short == 'MiniLM-L4-v2':
	base_accuracy = 0.7645  # from previous evaluation
elif model_name_short == 'MiniLM-L6-v2':
	base_accuracy = 0.7871
else:
	model = CrossEncoder(model_name)
	# Base accuracy using R-Precision without applying k+1 or a threshold
	base_accuracy = evaluate_model(model, model_name_short)

### Prepare the data

In [3]:
# Preprocess the training dataset
# Create triplets of (query, passage, score) for training
import csv

if 'train_data' not in globals():  # or 'validation_data' not in globals():
	train_data_file = 'coding_train_data.csv'
	validation_data_file = 'coding_validation_data.csv'

	if os.path.exists(train_data_file):  # and os.path.exists(validation_data_file):
		with open(train_data_file, 'r') as f:
			train_data = [row for row in csv.reader(f)][1:]
		# with open(validation_data_file, 'r') as f:
		# 	validation_data = [row for row in csv.reader(f)][1:]
	else:
		raise NotImplementedError('Please upload the dataset files')
		# from datasets import load_from_disk
		# dataset = load_from_disk('coding_dataset')
		# train_dataset = dataset['train']
		# validation_dataset = dataset['validation']

		# train_data = []
		# for val_index, row in enumerate(train_dataset):
		# 	query = row['query']
		# 	passages = row['passages']['passage_text']
		# 	scores = row['passages']['is_selected']
		# 	for passage, score in zip(passages, scores):
		# 		train_data.append([query, passage, score])
		# with open(train_data_file, 'w', newline='') as f:
		# 	writer = csv.writer(f)
		# 	writer.writerow(['query', 'passage', 'score'])  # Write header
		# 	writer.writerows(train_data)

		# validation_data = []
		# for val_index, row in enumerate(validation_dataset):
		# 	query = row['query']
		# 	passages = row['passages']['passage_text']
		# 	scores = row['passages']['is_selected']
		# 	for passage, score in zip(passages, scores):
		# 		validation_data.append([query, passage, score])
		# with open(validation_data_file, 'w', newline='') as f:
		# 	writer = csv.writer(f)
		# 	writer.writerow(['query', 'passage', 'score'])  # Write header
		# 	writer.writerows(validation_data)

print('Sample:')
for query, passage, score in train_data[:1]:
	print(f'  Query: {query}  \n  Passage: {passage[:50]}...  \n  Score: {score}\n')
print(f'Train data size: {len(train_data)}')
# print(f'Validation data size: {len(validation_data)}')

Sample:
  Query: what is an of clause sql  
  Passage: SQL clauses site was designed to help programmers ...  
  Score: 0

Train data size: 25078


In [4]:
# train_data_file_jsonl = 'coding_train_data.jsonl'
# validation_data_file_jsonl = 'coding_validation_data.jsonl'

# import os
# import json

# # Function to create a JSONL file from a dataset - for DPO
# def create_jsonl(dataset, jsonl_path):
# 	with open(jsonl_path, 'w') as jsonl_file:
# 		for row in dataset:
# 			query = row['query']
# 			passages = row['passages']['passage_text']
# 			scores = row['passages']['is_selected']
# 			selected_passages = [passage for passage, score in zip(passages, scores) if score == 1]
# 			# If there are no selected passages, skip this row
# 			if not selected_passages:
# 				continue
# 			rejected_passages = [passage for passage, score in zip(passages, scores) if score == 0]

# 			for selected_passage in selected_passages:
# 				for rejected_passage in rejected_passages:
# 					# Create the JSON object with the expected keys
# 					json_obj = {
# 						'prompt': query,
# 						'chosen': selected_passage,
# 						'rejected': rejected_passage
# 					}
# 					jsonl_file.write(json.dumps(json_obj) + '\n')

# # Check if the JSONL files exist, and create them if they don't
# if not os.path.exists(train_data_file_jsonl):
# 	create_jsonl(train_dataset, train_data_file_jsonl)

# if not os.path.exists(validation_data_file_jsonl):
# 	create_jsonl(validation_dataset, validation_data_file_jsonl)

# # Convert the CSV files to make it work with `datasets` library
# from datasets import load_dataset

# data_files = {
# 	'train': train_data_file,  # num_rows: 25078
# 	'validation': validation_data_file,  # num_rows: 3093
# }

# coding_dataset_triplets = load_dataset(
# 	'csv',
# 	data_files=data_files,
# )
# coding_dataset_triplets.save_to_disk('coding_dataset_triplets')

### Cross-encoder fine-tuning

In [5]:
from sentence_transformers import InputExample
from torch.utils.data import DataLoader
import torch

train_samples = [InputExample(texts=[query, passage], label=float(score))
				 for query, passage, score in train_data]

model_save_path = f'msmarco-coding-{model_name_short}'

try:
	if not os.path.exists(model_save_path):
		raise FileNotFoundError(f'Model not found: {model_save_path}')
	print('Trying to load the model...')
	loaded_model = CrossEncoder(model_save_path)
	if loaded_model:
		model = loaded_model
	print(f'Model loaded. Skipping fine-tuning.')
except KeyboardInterrupt:
	print('KeyboardInterrupt: Stopping fine-tuning.')
	sys.exit(0)
except:
	print('Loading the model...')
	model = CrossEncoder(model_name)
	print('Fine-tuning the model...')
	train_dataloader = DataLoader(train_samples,
							batch_size=32, shuffle=True)
	num_epochs = 3  # 5 for L4 model

	model.fit(
		train_dataloader=train_dataloader,
		epochs=num_epochs,
		loss_fct=torch.nn.BCEWithLogitsLoss(),  # works for binary labels
		warmup_steps=int(len(train_dataloader) * num_epochs * 0.1),  # 10% of total steps
		optimizer_class=torch.optim.AdamW,
		optimizer_params={ 'lr': 2e-5 },  # learning rate
		use_amp=True,  # for mixed precision training such as fp16
		output_path=None,  # skip saving every run
	)

ft_accuracy = evaluate_model(model, model_name_short+'-fine-tuned')
if ft_accuracy > base_accuracy:
	print(f'GOOD. Fine-tuned model is better than the base model.')
	model.save(model_save_path)
	print(f'Model saved to {model_save_path}')
else:
	print(f'BAD. Fine-tuned model is worse than the base model.')
print(f'                  ({base_accuracy*100:.2f} -> {ft_accuracy*100:.2f})')

Trying to load the model...
Model loaded. Skipping fine-tuning.
Accuracy: 78.87%
Model: MiniLM-L6-v2-fine-tuned
GOOD. Fine-tuned model is better than the base model.
Model saved to msmarco-coding-MiniLM-L6-v2
                  (78.71 -> 78.87)


### Hyperparameter tuning

In [6]:
# import csv
# import itertools
# import math
# import pandas as pd
# import torch
# import torch.nn as nn
# import torch_optimizer

# result_filename = 'hyperparam_optim_results.csv'
# try:
# 	results = pd.read_csv(result_filename).to_dict(orient='records')
# 	# replace 'None' with None in all columns
# 	for result in results:
# 		for key, value in result.items():
# 			if value == None or (type(value) == float and math.isnan(value)):
# 				result[key] = None
# except:
# 	results = []

# def save_last_result():
# 	with open(result_filename, mode='a', newline='') as file:
# 		writer = csv.DictWriter(file, fieldnames=results[0].keys())
# 		if len(results) == 1:  # we are saving first result. write header
# 			writer.writeheader()
# 		writer.writerow(results[-1])

# 	if 'google.colab' in sys.modules:
# 		!cp *_results.csv drive/MyDrive/iRAT/

# param_grid = {
# 	'learn_rates':    [1e-5, 2e-5, 3e-5],
# 	'batch_size':     [32],
# 	'epochs':         [2, 3, 5],
# 	'optimizers':     [torch.optim.AdamW, torch_optimizer.AdamP],
# 	'loss_functions': [None, nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.0]))]
# 						# Hyperparam tuning is to find best non-default parameters.
# }
# evaluated_combinations = {tuple(result.values())[:-1] for result in results}
# 												# -1 to exclude 'accuracy' values

# total_combinations = 1
# for key in param_grid:
# 	total_combinations *= len(param_grid[key])

# pending = 0
# for learn_rate, batch_size, epochs, optimizer, loss_fct in itertools.product(
# 		param_grid['learn_rates'], param_grid['batch_size'], param_grid['epochs'],
# 		param_grid['optimizers'], param_grid['loss_functions']):
# 	# check whether the current combination has already been evaluated
# 	loss_fct_str = str(loss_fct) if loss_fct else None
# 	current_combination = (learn_rate, batch_size, epochs, optimizer.__name__, loss_fct_str)
# 	if current_combination in evaluated_combinations:
# 		continue
# 	pending += 1

# print(f'Pending combinations: {pending}')


# for learn_rate, batch_size, epochs, optimizer, loss_fct in itertools.product(
# 		param_grid['learn_rates'], param_grid['batch_size'], param_grid['epochs'],
# 		param_grid['optimizers'], param_grid['loss_functions']):
# 	# check whether the current combination has already been evaluated
# 	loss_fct_str = str(loss_fct) if loss_fct else None
# 	current_combination = (learn_rate, batch_size, epochs, optimizer.__name__, loss_fct_str)
# 	if current_combination in evaluated_combinations:
# 		print('Skipping already evaluated combination.')
# 		continue

# 	print(f'Index:', len(results)+1, 'of', total_combinations)
# 	print(current_combination)
# 	pending -= 1
# 	print(f'Pending after this: {pending}')
# 	train_dl = DataLoader(train_samples, batch_size=batch_size, shuffle=True)

# 	model = CrossEncoder(model_name)
# 	model.fit(
# 		train_dataloader=train_dl,
# 		epochs=epochs,
# 		loss_fct=loss_fct,
# 		optimizer_class=optimizer,
# 		optimizer_params={
# 			'lr': learn_rate,
# 		},
# 		warmup_steps=int(0.1 * len(train_dl) * epochs),
# 		use_amp=True,
# 		output_path=None,  # skip saving every run
# 		show_progress_bar=False,
# 	)

# 	accuracy = evaluate_model(model, model_name_short=None)
# 	results.append({
# 		'learn_rate': learn_rate,
# 		'batch_size': batch_size,
# 		'epochs': epochs,
# 		'optimizer': optimizer.__name__,
# 		'loss_function': loss_fct_str,
# 		'accuracy': accuracy
# 	})
# 	save_last_result()
# 	print('-'*80)

# df = pd.DataFrame(results)
# df.sort_values(by='accuracy', ascending=False, inplace=True)
# df.fillna(value='None', inplace=True)
# df.to_csv(result_filename, index=False)

# if 'google.colab' in sys.modules:
# 	!cp *_results.csv drive/MyDrive/iRAT/
# 	!ls drive/MyDrive/iRAT/*_results.csv  # ensure file is printed

# df